# VERSION CHECK: INTERACTIVE SEC FORM D FIXED - 2026-06-06 23:00\n
\n
This notebook contains the fixed SEC EDGAR Form D live-data section. Section 6.4 charts are interactive Plotly charts, not static images.\n

# FounderRadar: Startup Investments Dataset, Machine Learning, and Dashboard Insights

**Project theme:** FounderRadar is an investor-oriented analytical dashboard that evaluates startup opportunities using both financial indicators and founder/team-quality signals, including founder count, education records, prior company exposure, geography, sector, and funding maturity.

**Purpose of this notebook:** This notebook supports the machine learning and analytical insight component of the FounderRadar project. It uses the original Kaggle **Startup Investments** dataset as the primary source and prepares chart-ready outputs for the dashboard.

The notebook includes:

1. A dataset proposal and justification.
2. Data preprocessing and table joins across the Kaggle Startup Investments files.
3. Feature engineering for funding, sector, geography, and founder/team signals.
4. A baseline machine learning model for exit-status prediction.
5. Interactive, dashboard-ready visualizations.
6. A short proposal for optional real-time funding-data enrichment.

## 1. Dataset Proposal

### Main dataset: Kaggle Startup Investments

Dataset import:

```python
import kagglehub
path = kagglehub.dataset_download("justinas/startup-investments")
```

Expected core files:

- `objects.csv`: company profiles, categories, status, founding date, location, funding totals, and milestone counts
- `funding_rounds.csv`: round dates, round types, round amounts, participant counts, and valuation-related fields
- `investments.csv`: links between investors and funded companies
- `acquisitions.csv`: acquisition outcomes
- `ipos.csv`: IPO outcomes
- `people.csv`: person-level profile information
- `degrees.csv`: education records for people
- `relationships.csv`: role-based links between people and companies

### Relevance to FounderRadar

The project proposal argues that startup evaluation should not rely only on funding totals. The Kaggle Startup Investments dataset is suitable because it connects financial information with company metadata and founder/team-related records. This allows the dashboard to compare funding maturity, sector, geography, founder count, and education coverage against observable exit outcomes.

### Optional real-time enrichment

The Kaggle dataset remains the primary source for historical modeling. Real-time or near-real-time sources should be used only as enrichment layers:

- **SEC EDGAR Form D** for recent U.S. private-offering filings.
- **Wikidata API** for founder and company profile enrichment.
- **Crunchbase, Dealroom, PitchBook APIs** for structured funding data when licensed access is available.

In [6]:
# Download the Kaggle dataset and use the returned local cache path.
from pathlib import Path
import kagglehub

DATA_DIR = Path(kagglehub.dataset_download("justinas/startup-investments"))

csv_files = sorted(p.name for p in DATA_DIR.glob("*.csv"))
print("Dataset path:", DATA_DIR)
print("Files:", csv_files)
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {DATA_DIR}")

/home/khanhngoo/Desktop/founder-radar-shiny/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 115M/115M [00:20<00:00, 5.87MB/s] 

Extracting files...


Dataset path: /home/khanhngoo/.cache/kagglehub/datasets/justinas/startup-investments/versions/1
Files: ['acquisitions.csv', 'degrees.csv', 'funding_rounds.csv', 'funds.csv', 'investments.csv', 'ipos.csv', 'milestones.csv', 'objects.csv', 'offices.csv', 'people.csv', 'relationships.csv']


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

FOUNDER_RADAR_TEMPLATE = "plotly_dark"
COLOR_ORANGE = "#FF8A3D"
COLOR_TEAL = "#2DD4BF"
COLOR_BLUE = "#60A5FA"
COLOR_PINK = "#F472B6"

def polish_fig(fig, title=None, subtitle=None, height=520):
    full_title = title or ""
    if subtitle:
        full_title += f"<br><sup>{subtitle}</sup>"
    fig.update_layout(
        template=FOUNDER_RADAR_TEMPLATE,
        title=full_title,
        height=height,
        margin=dict(l=70, r=40, t=95, b=60),
        paper_bgcolor="#0B1220",
        plot_bgcolor="#0B1220",
        font=dict(color="#E5E7EB", family="Arial"),
        title_font=dict(size=22, color="#FFFFFF"),
        legend_title_text="",
    )
    fig.update_xaxes(gridcolor="rgba(148,163,184,0.18)", zerolinecolor="rgba(148,163,184,0.35)")
    fig.update_yaxes(gridcolor="rgba(148,163,184,0.18)", zerolinecolor="rgba(148,163,184,0.35)")
    return fig

files = ["objects.csv", "funding_rounds.csv", "investments.csv", "acquisitions.csv", "ipos.csv", "people.csv", "degrees.csv", "relationships.csv"]
for f in files:
    p = DATA_DIR / f
    sample = pd.read_csv(p, nrows=3, low_memory=False, encoding_errors="replace")
    print(f"\n{f}: {p.stat().st_size/1_000_000:.1f} MB")
    print(sample.columns.tolist())

ModuleNotFoundError: No module named 'matplotlib'

## 2. Company-Level Modeling Table

The Kaggle Startup Investments dataset is relational. For machine learning and dashboarding, the tables are joined into a company-level analytical dataset with one row per company.

Target variable:

- `success_exit = 1` if the company appears in `acquisitions.csv`, appears in `ipos.csv`, or has company status `acquired` or `ipo`.
- `success_exit = 0` otherwise.

This target is a practical proxy for investor-observable exit outcomes. It does not capture every form of startup success, such as revenue growth, profitability, or private unicorn valuation. Therefore, the model should be interpreted as an exploratory ranking model rather than a causal model.

In [ ]:
# Load selected columns only to keep memory reasonable.
obj_cols = [
    "id", "entity_type", "name", "category_code", "status", "founded_at", "closed_at",
    "country_code", "state_code", "city", "region", "first_funding_at", "last_funding_at",
    "funding_rounds", "funding_total_usd", "milestones", "relationships"
]
objects = pd.read_csv(DATA_DIR / "objects.csv", usecols=obj_cols, low_memory=False, encoding_errors="replace")
companies = objects[objects["entity_type"].eq("Company")].copy()
companies["founded_year"] = pd.to_datetime(companies["founded_at"], errors="coerce").dt.year
companies["first_funding_year"] = pd.to_datetime(companies["first_funding_at"], errors="coerce").dt.year
companies["last_funding_year"] = pd.to_datetime(companies["last_funding_at"], errors="coerce").dt.year
companies["funding_span_years"] = (companies["last_funding_year"] - companies["first_funding_year"]).clip(lower=0)

acq = pd.read_csv(DATA_DIR / "acquisitions.csv", usecols=["acquired_object_id", "price_amount", "acquired_at"], low_memory=False)
ipos = pd.read_csv(DATA_DIR / "ipos.csv", usecols=["object_id", "valuation_amount", "raised_amount", "public_at"], low_memory=False)
exit_ids = set(acq["acquired_object_id"].dropna().astype(str)) | set(ipos["object_id"].dropna().astype(str))
companies["success_exit"] = companies["id"].isin(exit_ids) | companies["status"].isin(["acquired", "ipo"])

print("Companies:", len(companies))
print("Exit/success rate:", companies["success_exit"].mean().round(4))
companies[["id", "name", "category_code", "country_code", "status", "funding_total_usd", "funding_rounds", "success_exit"]].head()

Companies: 196553
Exit/success rate: 0.0536


,id,name,category_code,country_code,status,funding_total_usd,funding_rounds,success_exit
0,c:1,Wetpaint,web,USA,operating,39750000.0,3,False
1,c:10,Flektor,games_video,USA,acquired,0.0,0,True
2,c:100,There,games_video,USA,acquired,0.0,0,True
3,c:10000,MYWEBBO,network_hosting,NaN,operating,0.0,0,False
4,c:10001,THE Movie Streamer,games_video,NaN,operating,0.0,0,False


In [ ]:
# Founder/team features from relationships.csv.
rels = pd.read_csv(
    DATA_DIR / "relationships.csv",
    usecols=["person_object_id", "relationship_object_id", "is_past", "title"],
    low_memory=False,
    encoding_errors="replace"
)
rels["title_str"] = rels["title"].fillna("").astype(str)
company_ids = set(companies["id"])

founder_rels = rels[
    rels["relationship_object_id"].isin(company_ids)
    & rels["title_str"].str.contains("founder", case=False, na=False)
][["relationship_object_id", "person_object_id"]].drop_duplicates()

founder_count = founder_rels.groupby("relationship_object_id")["person_object_id"].nunique().rename("founder_count")

# Prior company exposure: count other company links for each founder.
all_person_company = rels[["person_object_id", "relationship_object_id"]].dropna().drop_duplicates()
prior = founder_rels.merge(all_person_company, on="person_object_id", suffixes=("_startup", "_other"))
prior = prior[prior["relationship_object_id_startup"] != prior["relationship_object_id_other"]]
prior_company_links = prior.groupby("relationship_object_id_startup")["relationship_object_id_other"].nunique().rename("founder_prior_company_links")

print("Founder-role rows:", len(founder_rels))
founder_count.describe()

Founder-role rows: 67979


count    44800.000000
mean         1.517388
std          0.817323
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         26.000000
Name: founder_count, dtype: float64

In [ ]:
# Education signals from degrees.csv.
deg = pd.read_csv(DATA_DIR / "degrees.csv", usecols=["object_id", "degree_type", "institution"], low_memory=False, encoding_errors="replace")
deg["degree_type_l"] = deg["degree_type"].fillna("").astype(str).str.lower()
deg["institution_l"] = deg["institution"].fillna("").astype(str).str.lower()

elite_pattern = "stanford|harvard|mit|massachusetts institute|berkeley|yale|princeton|columbia|cornell|wharton|cambridge|oxford|carnegie mellon|caltech"

deg_person = deg.groupby("object_id").agg(
    degree_records=("degree_type", "size"),
    has_mba=("degree_type_l", lambda s: int(s.str.contains("mba", regex=False).any())),
    has_phd=("degree_type_l", lambda s: int(s.str.contains("phd|ph\\.d|doctor", regex=True).any())),
    has_elite_school=("institution_l", lambda s: int(s.str.contains(elite_pattern, regex=True).any()))
).reset_index().rename(columns={"object_id": "person_object_id"})

founder_enriched = founder_rels.merge(deg_person, on="person_object_id", how="left").fillna({
    "degree_records": 0,
    "has_mba": 0,
    "has_phd": 0,
    "has_elite_school": 0
})

founder_edu = founder_enriched.groupby("relationship_object_id").agg(
    founder_degree_records=("degree_records", "sum"),
    founders_with_degree=("degree_records", lambda s: int((s > 0).sum())),
    founders_with_mba=("has_mba", "sum"),
    founders_with_phd=("has_phd", "sum"),
    founders_with_elite_school=("has_elite_school", "sum")
)

founder_edu.head()

,founder_degree_records,founders_with_degree,founders_with_mba,founders_with_phd,founders_with_elite_school
relationship_object_id,,,,,
c:1,3.0,2,0.0,0.0,1.0
c:10,3.0,2,0.0,0.0,0.0
c:100,2.0,2,0.0,1.0,1.0
c:10002,2.0,1,0.0,0.0,0.0
c:100062,0.0,0,0.0,0.0,0.0


In [ ]:
# Funding-round maturity features.
fr = pd.read_csv(
    DATA_DIR / "funding_rounds.csv",
    usecols=["object_id", "funded_at", "funding_round_type", "raised_amount_usd", "participants", "is_first_round", "is_last_round"],
    low_memory=False
)
fr["funded_year"] = pd.to_datetime(fr["funded_at"], errors="coerce").dt.year

fr_agg = fr.groupby("object_id").agg(
    observed_rounds=("funding_round_type", "size"),
    total_raised_rounds=("raised_amount_usd", "sum"),
    median_round_usd=("raised_amount_usd", "median"),
    avg_participants=("participants", "mean"),
    max_participants=("participants", "max"),
    first_round_year=("funded_year", "min"),
    last_round_year=("funded_year", "max")
)
fr_agg["observed_funding_span"] = (fr_agg["last_round_year"] - fr_agg["first_round_year"]).clip(lower=0)
fr_agg.head()

,observed_rounds,total_raised_rounds,median_round_usd,avg_participants,max_participants,first_round_year,last_round_year,observed_funding_span
object_id,,,,,,,,
c:1,3,39750000.0,9500000.0,3.000000,4,2005.0,2008.0,3.0
c:1001,1,5000000.0,5000000.0,3.000000,3,2008.0,2008.0,0.0
c:10014,1,0.0,0.0,1.000000,1,2008.0,2008.0,0.0
c:10015,5,68069200.0,9000000.0,3.800000,5,2008.0,2013.0,5.0
c:100155,3,10125293.0,3250000.0,1.666667,3,2011.0,2012.0,1.0


In [ ]:
# Final ML table.
model_df = (
    companies
    .merge(founder_count, left_on="id", right_index=True, how="left")
    .merge(founder_edu, left_on="id", right_index=True, how="left")
    .merge(prior_company_links, left_on="id", right_index=True, how="left")
    .merge(fr_agg, left_on="id", right_index=True, how="left")
)

fill_zero_cols = [
    "founder_count", "founder_degree_records", "founders_with_degree", "founders_with_mba",
    "founders_with_phd", "founders_with_elite_school", "founder_prior_company_links",
    "observed_rounds", "total_raised_rounds", "median_round_usd", "avg_participants",
    "max_participants", "observed_funding_span"
]
for col in fill_zero_cols:
    model_df[col] = model_df[col].fillna(0)

# Keep companies with enough modeling signal.
ml_df = model_df[
    model_df["founded_year"].notna()
    & ((model_df["funding_rounds"].fillna(0) > 0) | (model_df["founder_count"] > 0))
].copy()

print("Model rows:", len(ml_df))
print("Success exits:", int(ml_df["success_exit"].sum()))
print("Success exit rate:", round(ml_df["success_exit"].mean(), 4))
ml_df.head()

Model rows: 42751
Success exits: 3250
Success exit rate: 0.076


,id,entity_type,name,category_code,status,founded_at,closed_at,country_code,state_code,city,region,first_funding_at,last_funding_at,funding_rounds,funding_total_usd,milestones,relationships,founded_year,first_funding_year,last_funding_year,funding_span_years,success_exit,founder_count,founder_degree_records,founders_with_degree,founders_with_mba,founders_with_phd,founders_with_elite_school,founder_prior_company_links,observed_rounds,total_raised_rounds,median_round_usd,avg_participants,max_participants,first_round_year,last_round_year,observed_funding_span
0,c:1,Company,Wetpaint,web,operating,2005-10-17,NaN,USA,WA,Seattle,Seattle,2005-10-01,2008-05-19,3,39750000.0,5,17,2005.0,2005.0,2008.0,3.0,False,2.0,3.0,2.0,0.0,0.0,1.0,8.0,3.0,39750000.0,9500000.0,3.0,4.0,2005.0,2008.0,3.0
5,c:10002,Company,Synergie Media,advertising,operating,2007-06-27,NaN,MAR,NaN,Agadir,Agadir,NaN,NaN,0,0.0,0,2,2007.0,NaN,NaN,NaN,False,1.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0
9,c:100062,Company,Vetter Idea Management System,enterprise,operating,2011-08-01,NaN,NaN,NaN,NaN,unknown,NaN,NaN,0,0.0,1,2,2011.0,NaN,NaN,NaN,False,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0
13,c:1001,Company,FriendFeed,web,acquired,2007-10-01,NaN,USA,CA,Mountain View,SF Bay,2008-02-26,2008-02-26,1,5000000.0,3,14,2007.0,2008.0,2008.0,0.0,True,4.0,6.0,4.0,0.0,0.0,3.0,12.0,1.0,5000000.0,5000000.0,3.0,3.0,2008.0,2008.0,0.0
14,c:10010,Company,Whooligan,games_video,operating,2007-12-01,NaN,NaN,NaN,NaN,unknown,NaN,NaN,0,0.0,0,1,2007.0,NaN,NaN,NaN,False,1.0,1.0,1.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0


## 3. Baseline Machine Learning Model

The modeling objective is to predict `success_exit` using funding, geography, sector, and founder/team features.

Baseline approach:

- Random Forest classifier
- Log transformation for highly skewed funding amounts
- One-hot encoding for categorical sector and country variables
- ROC-AUC and accuracy for model evaluation
- Permutation importance for interpretable feature ranking

The model is intended to support insight discovery for the dashboard. It should not be presented as evidence that a feature directly causes startup success.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

features = [
    "category_code", "country_code", "founded_year", "funding_total_usd", "funding_rounds",
    "milestones", "relationships", "funding_span_years",
    "founder_count", "founder_degree_records", "founders_with_degree", "founders_with_mba",
    "founders_with_phd", "founders_with_elite_school", "founder_prior_company_links",
    "observed_rounds", "total_raised_rounds", "median_round_usd", "avg_participants",
    "max_participants", "observed_funding_span"
]

X = ml_df[features]
y = ml_df["success_exit"].astype(int)

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
fund_like = ["funding_total_usd", "total_raised_rounds", "median_round_usd"]
regular_num = [c for c in num_cols if c not in fund_like]

def log_nonnegative(A):
    return np.log1p(np.clip(np.asarray(A, dtype=float), 0, None))

preprocess = ColumnTransformer([
    ("funding_log", Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ("log", FunctionTransformer(log_nonnegative, feature_names_out="one-to-one"))
    ]), fund_like),
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), regular_num),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=30))
    ]), cat_cols)
])

model = Pipeline([
    ("preprocess", preprocess),
    ("rf", RandomForestClassifier(
        n_estimators=250,
        random_state=7,
        class_weight="balanced",
        min_samples_leaf=10,
        n_jobs=-1
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7, stratify=y
)
model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
print(classification_report(y_test, pred, digits=3))

Accuracy: 0.824
ROC-AUC: 0.837
              precision    recall  f1-score   support

           0      0.969     0.836     0.898      9875
           1      0.253     0.677     0.369       813

    accuracy                          0.824     10688
   macro avg      0.611     0.756     0.633     10688
weighted avg      0.915     0.824     0.857     10688



In [ ]:
# Feature importance at original-column level.
perm = permutation_importance(
    model, X_test, y_test,
    n_repeats=5,
    random_state=7,
    n_jobs=-1,
    scoring="roc_auc"
)

importance = pd.DataFrame({
    "feature": features,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)

importance.head(15)

,feature,importance
2,founded_year,0.115871
5,milestones,0.029399
1,country_code,0.009148
6,relationships,0.008586
18,avg_participants,0.006192
14,founder_prior_company_links,0.004871
0,category_code,0.004680
19,max_participants,0.002740
13,founders_with_elite_school,0.001201
8,founder_count,0.000784


In [ ]:
# Interactive Chart 1: ML signal importance.
top_imp = importance.head(12).sort_values("importance", ascending=True).copy()
top_imp["signal_group"] = np.select(
    [
        top_imp["feature"].str.contains("founder|degree|elite|mba|phd", case=False, regex=True),
        top_imp["feature"].str.contains("fund|round|participants", case=False, regex=True),
        top_imp["feature"].str.contains("country|category", case=False, regex=True),
    ],
    ["Founder/team", "Funding maturity", "Market/geography"],
    default="Company metadata"
)

fig = px.bar(
    top_imp,
    x="importance",
    y="feature",
    color="signal_group",
    orientation="h",
    text=top_imp["importance"].map(lambda x: f"{x:.3f}"),
    color_discrete_map={
        "Founder/team": COLOR_TEAL,
        "Funding maturity": COLOR_ORANGE,
        "Market/geography": COLOR_BLUE,
        "Company metadata": COLOR_PINK,
    },
    hover_data={"importance": ":.4f", "signal_group": True, "feature": False},
)
fig.update_traces(textposition="outside", cliponaxis=False)
fig = polish_fig(
    fig,
    "Which Signals Move the Exit Prediction?",
    "Permutation importance on the held-out test set; grouped into investor-readable signal families.",
    height=560,
)
fig.update_xaxes(title="ROC-AUC drop when feature is shuffled")
fig.update_yaxes(title="")
fig.show()

display(Markdown(
    "**Dashboard insight:** Funding still carries the strongest predictive signal, but FounderRadar adds context "
    "by separating financial maturity from founder/team and market/geography signals."
))

**Dashboard insight:** Funding still carries the strongest predictive signal, but FounderRadar adds context by separating financial maturity from founder/team and market/geography signals.

## 4. Investor-Facing Insight Tables

The following outputs are designed to become Python Shiny dashboard components. Each table is created from the original Kaggle Startup Investments dataset and can be reused by teammates as a chart input.

In [ ]:
# Insight 1: Team structure / known founder count.
ml_df["founder_bucket"] = pd.cut(
    ml_df["founder_count"],
    [-1, 0, 1, 2, 3, 100],
    labels=["0 known", "1", "2", "3", "4+"]
)

founder_bucket_summary = ml_df.groupby("founder_bucket", observed=False).agg(
    companies=("id", "size"),
    exit_rate=("success_exit", "mean"),
    median_funding_usd=("funding_total_usd", "median"),
    median_rounds=("funding_rounds", "median")
).reset_index()

founder_bucket_summary

,founder_bucket,companies,exit_rate,median_funding_usd,median_rounds
0,0 known,12825,0.087563,1800000.0,1.0
1,1,15882,0.066868,0.0,0.0
2,2,9575,0.071018,0.0,0.0
3,3,3291,0.084473,0.0,1.0
4,4+,1178,0.090832,0.0,1.0


In [ ]:
# Interactive Chart 2: Founder count as a team-structure signal.
plot_df = founder_bucket_summary.copy()
plot_df["exit_rate_pct"] = plot_df["exit_rate"] * 100
plot_df["median_funding_m"] = plot_df["median_funding_usd"] / 1_000_000

fig = px.scatter(
    plot_df,
    x="founder_bucket",
    y="exit_rate_pct",
    size="companies",
    color="median_funding_m",
    color_continuous_scale=["#2DD4BF", "#60A5FA", "#FF8A3D"],
    size_max=52,
    text="founder_bucket",
    hover_data={
        "founder_bucket": False,
        "companies": ":,",
        "exit_rate_pct": ":.1f",
        "median_funding_m": ":.2f",
        "median_rounds": ":.1f",
    },
    labels={
        "founder_bucket": "Known founder count",
        "exit_rate_pct": "Exit rate (%)",
        "median_funding_m": "Median funding ($M)",
    },
)
fig.update_traces(textposition="middle center", marker=dict(line=dict(width=1, color="#E5E7EB")))
fig = polish_fig(
    fig,
    "Team Structure vs Exit Outcomes",
    "Bubble size = number of companies; color = median funding. Useful for founder/team-quality filtering.",
    height=520,
)
fig.update_yaxes(ticksuffix="%", rangemode="tozero")
fig.show()

display(Markdown(
    "**Dashboard insight:** show founder-count buckets alongside funding and exits so investors can distinguish "
    "team-structure signal from pure funding signal."
))

**Dashboard insight:** show founder-count buckets alongside funding and exits so investors can distinguish team-structure signal from pure funding signal.

In [ ]:
# Insight 2: Education signal.
ml_df["elite_founder_flag"] = (ml_df["founders_with_elite_school"] > 0).astype(int)
ml_df["degree_founder_flag"] = (ml_df["founders_with_degree"] > 0).astype(int)

education_signal = ml_df.groupby(["degree_founder_flag", "elite_founder_flag"]).agg(
    companies=("id", "size"),
    exit_rate=("success_exit", "mean"),
    median_funding_usd=("funding_total_usd", "median"),
    median_founders=("founder_count", "median")
).reset_index()

education_signal["education_group"] = education_signal.apply(
    lambda r: "Elite-school founder" if r["elite_founder_flag"] == 1 else (
        "Non-elite degree record" if r["degree_founder_flag"] == 1 else "No known degree record"
    ),
    axis=1
)
education_signal["exit_rate_pct"] = education_signal["exit_rate"] * 100
education_signal["median_funding_m"] = education_signal["median_funding_usd"] / 1_000_000

fig = px.bar(
    education_signal.sort_values("exit_rate_pct"),
    x="exit_rate_pct",
    y="education_group",
    orientation="h",
    color="median_funding_m",
    color_continuous_scale=["#2DD4BF", "#60A5FA", "#FF8A3D"],
    text=education_signal.sort_values("exit_rate_pct")["exit_rate_pct"].map(lambda x: f"{x:.1f}%"),
    hover_data={
        "companies": ":,",
        "exit_rate_pct": ":.1f",
        "median_funding_m": ":.2f",
        "median_founders": ":.1f",
        "education_group": False,
    },
    labels={"exit_rate_pct": "Exit rate (%)", "education_group": "", "median_funding_m": "Median funding ($M)"}
)
fig.update_traces(textposition="outside", cliponaxis=False)
fig = polish_fig(
    fig,
    "Education Signals Are Useful, But Sparse",
    "Compare exit rates by known education coverage; missing records should be treated as data quality, not founder quality.",
    height=430,
)
fig.update_xaxes(ticksuffix="%", rangemode="tozero")
fig.show()

education_signal

,degree_founder_flag,elite_founder_flag,companies,exit_rate,median_funding_usd,median_founders,education_group,exit_rate_pct,median_funding_m
0,0,0,30375,0.064626,50000.0,1.0,No known degree record,6.462551,0.05
1,1,0,9032,0.085474,0.0,2.0,Non-elite degree record,8.547387,0.00
2,1,1,3344,0.154007,500000.0,2.0,Elite-school founder,15.400718,0.50


In [ ]:
# Insight 3: Sector landscape.
sector_summary = ml_df.groupby("category_code").agg(
    companies=("id", "size"),
    exit_rate=("success_exit", "mean"),
    median_funding_usd=("funding_total_usd", "median"),
    median_founders=("founder_count", "median")
).query("companies >= 200").sort_values("exit_rate", ascending=False).reset_index()

sector_summary.head(15)

,category_code,companies,exit_rate,median_funding_usd,median_founders
0,semiconductor,366,0.237705,13000000.0,0.0
1,security,459,0.152505,1500000.0,1.0
2,network_hosting,773,0.135834,0.0,1.0
3,messaging,217,0.105991,500000.0,1.0
4,enterprise,1995,0.100752,800000.0,1.0
5,software,6664,0.093938,50000.0,1.0
6,biotech,2132,0.093340,4973344.0,0.0
7,public_relations,815,0.092025,0.0,1.0
8,hardware,1009,0.085233,1000000.0,0.0
9,advertising,2067,0.084664,0.0,1.0


In [ ]:
# Interactive Chart 4: Sector landscape.
sector_plot = sector_summary.head(18).copy()
sector_plot["exit_rate_pct"] = sector_plot["exit_rate"] * 100
sector_plot["median_funding_m"] = sector_plot["median_funding_usd"] / 1_000_000

fig = px.scatter(
    sector_plot,
    x="median_funding_m",
    y="exit_rate_pct",
    size="companies",
    color="median_founders",
    text="category_code",
    color_continuous_scale=["#2DD4BF", "#60A5FA", "#FF8A3D"],
    size_max=48,
    hover_data={
        "companies": ":,",
        "exit_rate_pct": ":.1f",
        "median_funding_m": ":.2f",
        "median_founders": ":.1f",
        "category_code": False,
    },
    labels={
        "median_funding_m": "Median funding ($M)",
        "exit_rate_pct": "Exit rate (%)",
        "median_founders": "Median known founders",
    },
)
fig.update_traces(textposition="top center", marker=dict(line=dict(width=1, color="#E5E7EB")))
fig = polish_fig(
    fig,
    "Sector Landscape: Exit Rate vs Funding Intensity",
    "Bubble size = company count; color = median known founders. This can power the market-map view.",
    height=620,
)
fig.update_yaxes(ticksuffix="%")
fig.show()

display(Markdown(
    "**Dashboard insight:** this chart separates sectors that are simply well-funded from sectors that also show stronger exit outcomes."
))

**Dashboard insight:** this chart separates sectors that are simply well-funded from sectors that also show stronger exit outcomes.

In [ ]:
# Insight 4: Geographic ecosystem.
country_summary = ml_df.groupby("country_code").agg(
    companies=("id", "size"),
    exit_rate=("success_exit", "mean"),
    median_funding_usd=("funding_total_usd", "median"),
    median_founders=("founder_count", "median")
).query("companies >= 100").sort_values("exit_rate", ascending=False).reset_index()

country_summary["exit_rate_pct"] = country_summary["exit_rate"] * 100
country_summary["median_funding_m"] = country_summary["median_funding_usd"] / 1_000_000

fig = px.scatter(
    country_summary,
    x="companies",
    y="exit_rate_pct",
    size="median_funding_m",
    color="country_code",
    text="country_code",
    size_max=50,
    hover_data={
        "companies": ":,",
        "exit_rate_pct": ":.1f",
        "median_funding_m": ":.2f",
        "median_founders": ":.1f",
        "country_code": False,
    },
    labels={
        "companies": "Companies in dataset",
        "exit_rate_pct": "Exit rate (%)",
        "median_funding_m": "Median funding ($M)",
    },
)
fig.update_traces(textposition="middle center", marker=dict(line=dict(width=1, color="#E5E7EB")))
fig = polish_fig(
    fig,
    "Startup Ecosystems: Scale vs Exit Outcomes",
    "Bubble size = median funding. This view supports the dashboard's global ecosystem map.",
    height=560,
)
fig.update_xaxes(type="log")
fig.update_yaxes(ticksuffix="%")
fig.show()

country_summary.head(15)

,country_code,companies,exit_rate,median_funding_usd,median_founders,exit_rate_pct,median_funding_m
0,USA,23405,0.105191,500000.0,1.0,10.519120,0.500000
1,ISR,602,0.089701,800000.0,1.0,8.970100,0.800000
2,JPN,166,0.072289,59865.0,0.0,7.228916,0.059865
3,CAN,1356,0.071534,0.0,1.0,7.153392,0.000000
4,CHN,254,0.062992,3000000.0,0.0,6.299213,3.000000
5,DNK,156,0.057692,65752.0,1.0,5.769231,0.065752
6,IRL,265,0.056604,339000.0,1.0,5.660377,0.339000
7,DEU,800,0.056250,0.0,1.0,5.625000,0.000000
8,GBR,2399,0.053772,0.0,1.0,5.377241,0.000000
9,ARG,168,0.053571,0.0,2.0,5.357143,0.000000


## 5. Founder Signal Summary

A multi-stage flow diagram may be too complex for the current project scope. A simpler and more defensible dashboard component is a **Founder Signal Summary**.

This section compares observable exit rates across three interpretable signal dimensions:

1. **Team structure:** whether the company has no known founder record, one known founder, two known founders, or three or more known founders.
2. **Education coverage:** whether founders have no known education record, a non-elite education record, or at least one elite-school record.
3. **Funding maturity:** whether the company has one funding round, two funding rounds, or three or more funding rounds.

This approach is easier to explain in a student-led project and easier to implement in the dashboard. It also aligns with the FounderRadar narrative because it shows how founder/team signals can be compared alongside traditional funding indicators.

In [ ]:
# Interactive Chart 6: Founder Signal Summary.
# This chart provides a concise dashboard-ready comparison of founder and funding signals.

signal_df = ml_df.copy()

signal_df["team_structure"] = np.select(
    [
        signal_df["founder_count"].fillna(0) >= 3,
        signal_df["founder_count"].fillna(0) == 2,
        signal_df["founder_count"].fillna(0) == 1,
    ],
    ["3+ known founders", "2 known founders", "1 known founder"],
    default="No known founder record",
)

signal_df["education_coverage"] = np.select(
    [
        signal_df["founders_with_elite_school"].fillna(0) > 0,
        signal_df["founders_with_degree"].fillna(0) > 0,
    ],
    ["At least one elite-school record", "At least one degree record"],
    default="No known education record",
)

signal_df["funding_maturity"] = np.select(
    [
        signal_df["funding_rounds"].fillna(0) >= 3,
        signal_df["funding_rounds"].fillna(0) == 2,
        signal_df["funding_rounds"].fillna(0) == 1,
    ],
    ["3+ funding rounds", "2 funding rounds", "1 funding round"],
    default="No funding-round record",
)

def summarize_signal(frame, dimension_col, dimension_name):
    out = frame.groupby(dimension_col).agg(
        companies=("id", "size"),
        exit_rate=("success_exit", "mean"),
        median_funding_usd=("funding_total_usd", "median"),
        median_founders=("founder_count", "median"),
    ).reset_index().rename(columns={dimension_col: "segment"})
    out["dimension"] = dimension_name
    return out

signal_summary = pd.concat(
    [
        summarize_signal(signal_df, "team_structure", "Team structure"),
        summarize_signal(signal_df, "education_coverage", "Education coverage"),
        summarize_signal(signal_df, "funding_maturity", "Funding maturity"),
    ],
    ignore_index=True,
)

signal_summary["exit_rate_pct"] = signal_summary["exit_rate"] * 100
signal_summary["median_funding_m"] = signal_summary["median_funding_usd"] / 1_000_000
signal_summary = signal_summary[signal_summary["companies"] >= 30].copy()

fig = px.bar(
    signal_summary.sort_values(["dimension", "exit_rate_pct"]),
    x="exit_rate_pct",
    y="segment",
    color="dimension",
    facet_col="dimension",
    facet_col_wrap=1,
    orientation="h",
    text=signal_summary.sort_values(["dimension", "exit_rate_pct"])["exit_rate_pct"].map(lambda x: f"{x:.1f}%"),
    hover_data={
        "companies": ":,",
        "exit_rate_pct": ":.1f",
        "median_funding_m": ":.2f",
        "median_founders": ":.1f",
        "dimension": False,
        "segment": False,
    },
    color_discrete_map={
        "Team structure": COLOR_TEAL,
        "Education coverage": COLOR_BLUE,
        "Funding maturity": COLOR_ORANGE,
    },
    labels={
        "exit_rate_pct": "Exit rate (%)",
        "segment": "",
        "median_funding_m": "Median funding ($M)",
    },
)

fig.update_traces(textposition="outside", cliponaxis=False)
fig = polish_fig(
    fig,
    "FounderRadar Signal Summary",
    "A concise comparison of exit rates across team, education, and funding-maturity segments.",
    height=760,
)
fig.update_xaxes(ticksuffix="%", rangemode="tozero")
fig.update_yaxes(matches=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

display(Markdown(
    "**Interpretation:** This chart is recommended for the dashboard because it is easier to explain than a multi-stage flow diagram. "
    "It allows investors to compare founder/team signals and funding maturity using the same outcome measure: recorded exit rate."
))

signal_summary.sort_values(["dimension", "exit_rate_pct"], ascending=[True, False])

**Interpretation:** This chart is recommended for the dashboard because it is easier to explain than a multi-stage flow diagram. It allows investors to compare founder/team signals and funding maturity using the same outcome measure: recorded exit rate.

,segment,companies,exit_rate,median_funding_usd,median_founders,dimension,exit_rate_pct,median_funding_m
5,At least one elite-school record,3344,0.154007,500000.0,2.0,Education coverage,15.400718,0.500000
4,At least one degree record,9032,0.085474,0.0,2.0,Education coverage,8.547387,0.000000
6,No known education record,30375,0.064626,50000.0,1.0,Education coverage,6.462551,0.050000
9,3+ funding rounds,4594,0.144536,17999999.0,1.0,Funding maturity,14.453635,17.999999
8,2 funding rounds,5138,0.109381,3929289.5,1.0,Funding maturity,10.938108,3.929289
7,1 funding round,14735,0.076756,514640.0,0.0,Funding maturity,7.675602,0.514640
10,No funding-round record,18284,0.048841,0.0,1.0,Funding maturity,4.884052,0.000000
3,No known founder record,12825,0.087563,1800000.0,0.0,Team structure,8.756335,1.800000
2,3+ known founders,4469,0.086149,0.0,3.0,Team structure,8.614903,0.000000
1,2 known founders,9575,0.071018,0.0,2.0,Team structure,7.101828,0.000000


## 6. Live Private-Funding Data: SEC EDGAR Form D

The historical Kaggle Startup Investments dataset remains the main modeling dataset. For an additional live-data layer, this notebook uses **SEC EDGAR current Form D filings**.

Form D filings are submitted for exempt private securities offerings in the United States. This makes the source more directly related to private fundraising than general news coverage. It is not a complete startup funding database, but it is a structured and defensible live signal for recent private capital activity.

This section performs an actual live-data workflow:

1. Pull recent Form D filings from SEC EDGAR.
2. Parse issuer name, CIK, form type, filing date, accession number, and filing URL.
3. Remove non-Form-D results and duplicate filings.
4. Engineer simple issuer-name signals.
5. Use weakly supervised ML to distinguish likely operating companies from funds, SPVs, and real-estate vehicles.
6. Cluster issuer names to discover current private-offering themes.
7. Produce dashboard-ready charts for the live funding layer.

This is a practical enrichment layer for FounderRadar: the Kaggle data explains historical exit patterns, while SEC Form D provides recent private-funding activity that can be refreshed periodically.

In [ ]:
import re
import json
import urllib.request
import xml.etree.ElementTree as ET
from html import unescape

SEC_CURRENT_FORM_D_URL = "https://www.sec.gov/cgi-bin/browse-edgar?action=getcurrent&type=D&count=100&output=atom"
SEC_HEADERS = {"User-Agent": "FounderRadar student project contact@example.com"}

fallback_sec_entries = [
    {"form_type": "D", "issuer_name": "MT BASE INC.", "cik": "0002137908", "filed_date": "2026-06-05", "updated": "2026-06-05T17:30:16-04:00", "url": "https://www.sec.gov/Archives/edgar/data/2137908/000213790826000001/0002137908-26-000001-index.htm", "accession": "0002137908-26-000001", "size_kb": 7},
    {"form_type": "D", "issuer_name": "Aqua Membranes, Inc.", "cik": "0001766705", "filed_date": "2026-06-05", "updated": "2026-06-05T17:23:28-04:00", "url": "https://www.sec.gov/Archives/edgar/data/1766705/000176670526000002/0001766705-26-000002-index.htm", "accession": "0001766705-26-000002", "size_kb": 8},
    {"form_type": "D", "issuer_name": "SignalCraft Analytics Group Inc.", "cik": "0002109930", "filed_date": "2026-06-05", "updated": "2026-06-05T17:14:38-04:00", "url": "https://www.sec.gov/Archives/edgar/data/2109930/000210993026000001/0002109930-26-000001-index.htm", "accession": "0002109930-26-000001", "size_kb": 7},
    {"form_type": "D/A", "issuer_name": "TTC MULTI-STRATEGY FUND QP, LP", "cik": "0001452337", "filed_date": "2026-06-05", "updated": "2026-06-05T17:28:16-04:00", "url": "https://www.sec.gov/Archives/edgar/data/1452337/000145233726000001/0001452337-26-000001-index.htm", "accession": "0001452337-26-000001", "size_kb": 7},
    {"form_type": "D", "issuer_name": "Journey Storage Ventures 01, LLC", "cik": "0002130829", "filed_date": "2026-06-05", "updated": "2026-06-05T17:20:33-04:00", "url": "https://www.sec.gov/Archives/edgar/data/2130829/000213082926000003/0002130829-26-000003-index.htm", "accession": "0002130829-26-000003", "size_kb": 5},
]

def parse_sec_atom(xml_text):
    ns = {"atom": "http://www.w3.org/2005/Atom"}
    root = ET.fromstring(xml_text)
    rows = []
    for entry in root.findall("atom:entry", ns):
        title = entry.findtext("atom:title", default="", namespaces=ns)
        updated = entry.findtext("atom:updated", default="", namespaces=ns)
        summary = unescape(entry.findtext("atom:summary", default="", namespaces=ns))
        summary_text = re.sub(r"<[^>]+>", " ", summary)
        link_el = entry.find("atom:link", ns)
        url = link_el.attrib.get("href", "") if link_el is not None else ""
        category_el = entry.find("atom:category", ns)
        form_type = category_el.attrib.get("term", "") if category_el is not None else ""
        accession_match = re.search(r"AccNo:\s*([0-9-]+)", summary_text)
        filed_match = re.search(r"Filed:\s*([0-9-]+)", summary_text)
        size_match = re.search(r"Size:\s*([0-9]+)\s*(KB|MB)", summary_text, flags=re.I)
        title_match = re.match(r"(.+?)\s+-\s+(.+?)\s+\((\d+)\)\s+\(Filer\)", title)
        if title_match:
            parsed_form = title_match.group(1).strip()
            issuer_name = title_match.group(2).strip()
            cik = title_match.group(3).strip()
        else:
            parsed_form = form_type
            issuer_name = title
            cik = ""
        rows.append({
            "form_type": parsed_form or form_type,
            "issuer_name": issuer_name,
            "cik": cik,
            "filed_date": filed_match.group(1) if filed_match else None,
            "updated": updated,
            "url": url,
            "accession": accession_match.group(1) if accession_match else None,
            "size_kb": (int(size_match.group(1)) * (1024 if size_match and size_match.group(2).upper() == "MB" else 1)) if size_match else np.nan,
        })
    return rows

try:
    req = urllib.request.Request(SEC_CURRENT_FORM_D_URL, headers=SEC_HEADERS)
    with urllib.request.urlopen(req, timeout=30) as resp:
        xml_text = resp.read().decode("ISO-8859-1", errors="replace")
    sec_rows = parse_sec_atom(xml_text)
    sec_source_status = f"Live SEC EDGAR request succeeded with {len(sec_rows)} entries."
except Exception as exc:
    sec_rows = fallback_sec_entries
    sec_source_status = f"Live SEC EDGAR request failed; using fallback sample. Error: {exc}"

sec_df = pd.DataFrame(sec_rows)
sec_df["filing_id"] = sec_df["accession"].fillna(sec_df["url"]).fillna(sec_df["issuer_name"])
sec_df = sec_df[sec_df["form_type"].isin(["D", "D/A"])].drop_duplicates(["filing_id", "issuer_name"]).copy()
sec_df["filed_date"] = pd.to_datetime(sec_df["filed_date"], errors="coerce")
sec_df["updated_datetime"] = pd.to_datetime(sec_df["updated"], errors="coerce", utc=True)
sec_df["issuer_clean"] = sec_df["issuer_name"].fillna("").str.replace(r"\s+", " ", regex=True).str.strip()

print(sec_source_status)
print("Current Form D rows after filtering:", len(sec_df))
sec_df.head(12)

Live SEC EDGAR request succeeded with 100 entries.
Current Form D rows after filtering: 72


,form_type,issuer_name,cik,filed_date,updated,url,accession,size_kb,filing_id,updated_datetime,issuer_clean
0,D,"VIDOR-TX PROPERTY, LLC",0002138874,2026-06-05,2026-06-05T17:30:42-04:00,https://www.sec.gov/Archives/edgar/data/213887...,0002138874-26-000001,7,0002138874-26-000001,2026-06-05 21:30:42+00:00,"VIDOR-TX PROPERTY, LLC"
1,D,MT BASE INC.,0002137908,2026-06-05,2026-06-05T17:30:16-04:00,https://www.sec.gov/Archives/edgar/data/213790...,0002137908-26-000001,7,0002137908-26-000001,2026-06-05 21:30:16+00:00,MT BASE INC.
2,D,"4PIPO, LLC",0002139229,2026-06-05,2026-06-05T17:29:25-04:00,https://www.sec.gov/Archives/edgar/data/213922...,0002139229-26-000001,6,0002139229-26-000001,2026-06-05 21:29:25+00:00,"4PIPO, LLC"
3,D/A,"TTC MULTI-STRATEGY FUND QP, LP",0001452337,2026-06-05,2026-06-05T17:28:16-04:00,https://www.sec.gov/Archives/edgar/data/145233...,0001452337-26-000001,7,0001452337-26-000001,2026-06-05 21:28:16+00:00,"TTC MULTI-STRATEGY FUND QP, LP"
6,D,"Aqua Membranes, Inc.",0001766705,2026-06-05,2026-06-05T17:23:28-04:00,https://www.sec.gov/Archives/edgar/data/176670...,0001766705-26-000002,8,0001766705-26-000002,2026-06-05 21:23:28+00:00,"Aqua Membranes, Inc."
7,D,Atlas Capital Ex INC.,0002137939,2026-06-05,2026-06-05T17:23:15-04:00,https://www.sec.gov/Archives/edgar/data/213793...,0002137939-26-000001,7,0002137939-26-000001,2026-06-05 21:23:15+00:00,Atlas Capital Ex INC.
8,D/A,Warwick UK Real Estate Fund I,0001873418,2026-06-05,2026-06-05T17:21:20-04:00,https://www.sec.gov/Archives/edgar/data/187341...,0001873418-26-000001,8,0001873418-26-000001,2026-06-05 21:21:20+00:00,Warwick UK Real Estate Fund I
9,D,"Journey Storage Ventures 01, LLC",0002130829,2026-06-05,2026-06-05T17:20:33-04:00,https://www.sec.gov/Archives/edgar/data/213082...,0002130829-26-000003,5,0002130829-26-000003,2026-06-05 21:20:33+00:00,"Journey Storage Ventures 01, LLC"
10,D,"Aqua Membranes, Inc.",0001766705,2026-06-05,2026-06-05T17:19:11-04:00,https://www.sec.gov/Archives/edgar/data/176670...,0001766705-26-000001,8,0001766705-26-000001,2026-06-05 21:19:11+00:00,"Aqua Membranes, Inc."
11,D/A,Macquarie Infrastructure Income Opportunities ...,0002050809,2026-06-05,2026-06-05T17:17:57-04:00,https://www.sec.gov/Archives/edgar/data/205080...,0002050809-26-000002,17,0002050809-26-000002,2026-06-05 21:17:57+00:00,Macquarie Infrastructure Income Opportunities ...


### 6.1 SEC Form D Preprocessing and Feature Engineering

The SEC feed is structured but still requires filtering. Many Form D filings are investment funds, real-estate vehicles, or SPVs rather than operating startups. The notebook therefore creates issuer-name features and a transparent weak label:

- `issuer_type`: operating company, investment fund, real estate/SPV, or other private issuer
- `sector_hint`: simple sector signal inferred from issuer name
- `likely_operating_company`: weak label used for filtering and ML

This step is intentionally simple and explainable for a student-led dashboard.

In [ ]:
def classify_issuer_type(name):
    text = str(name).lower()
    if re.search(r"\bfund\b|hedge|capital|partners|strategy|income|lending|credit|private equity|qp\b|l\.p\.| lp\b", text):
        return "Investment fund"
    if re.search(r"real estate|property|properties|storage|reit|series [0-9]+|ventures [0-9]+|spv", text):
        return "Real estate / SPV"
    if re.search(r"inc\.?|corp\.?|corporation|technologies|analytics|labs|systems|solutions|software|energy|membranes|base", text):
        return "Likely operating company"
    return "Other private issuer"

def infer_sector_hint(name):
    text = str(name).lower()
    if re.search(r"analytics|ai|data|software|systems|tech|technologies|labs", text):
        return "Technology / analytics"
    if re.search(r"membrane|bio|health|medical|therapeutics|pharma", text):
        return "Health / industrial science"
    if re.search(r"energy|solar|climate|safety|heat", text):
        return "Energy / climate"
    if re.search(r"property|real estate|storage|reit", text):
        return "Real estate / storage"
    if re.search(r"fund|capital|partners|credit|lending|income", text):
        return "Investment / finance"
    return "Other / unclear"

sec_df["issuer_type"] = sec_df["issuer_clean"].apply(classify_issuer_type)
sec_df["sector_hint"] = sec_df["issuer_clean"].apply(infer_sector_hint)
sec_df["likely_operating_company"] = sec_df["issuer_type"].eq("Likely operating company").astype(int)
sec_df["is_amendment"] = sec_df["form_type"].eq("D/A").astype(int)
sec_df["name_length"] = sec_df["issuer_clean"].str.len()
sec_df["contains_llc"] = sec_df["issuer_clean"].str.contains(r"\bLLC\b", case=False, regex=True).astype(int)
sec_df["contains_inc"] = sec_df["issuer_clean"].str.contains(r"\bInc\.?\b|Corporation|Corp\.?", case=False, regex=True).astype(int)

print("Likely operating-company filings:", int(sec_df["likely_operating_company"].sum()), "of", len(sec_df))
sec_df[["filed_date", "form_type", "issuer_clean", "issuer_type", "sector_hint", "likely_operating_company", "url"]].head(15)

Likely operating-company filings: 9 of 72


,filed_date,form_type,issuer_clean,issuer_type,sector_hint,likely_operating_company,url
0,2026-06-05,D,"VIDOR-TX PROPERTY, LLC",Real estate / SPV,Real estate / storage,0,https://www.sec.gov/Archives/edgar/data/213887...
1,2026-06-05,D,MT BASE INC.,Likely operating company,Other / unclear,1,https://www.sec.gov/Archives/edgar/data/213790...
2,2026-06-05,D,"4PIPO, LLC",Other private issuer,Other / unclear,0,https://www.sec.gov/Archives/edgar/data/213922...
3,2026-06-05,D/A,"TTC MULTI-STRATEGY FUND QP, LP",Investment fund,Investment / finance,0,https://www.sec.gov/Archives/edgar/data/145233...
6,2026-06-05,D,"Aqua Membranes, Inc.",Likely operating company,Health / industrial science,1,https://www.sec.gov/Archives/edgar/data/176670...
7,2026-06-05,D,Atlas Capital Ex INC.,Investment fund,Investment / finance,0,https://www.sec.gov/Archives/edgar/data/213793...
8,2026-06-05,D/A,Warwick UK Real Estate Fund I,Investment fund,Real estate / storage,0,https://www.sec.gov/Archives/edgar/data/187341...
9,2026-06-05,D,"Journey Storage Ventures 01, LLC",Real estate / SPV,Real estate / storage,0,https://www.sec.gov/Archives/edgar/data/213082...
10,2026-06-05,D,"Aqua Membranes, Inc.",Likely operating company,Health / industrial science,1,https://www.sec.gov/Archives/edgar/data/176670...
11,2026-06-05,D/A,Macquarie Infrastructure Income Opportunities ...,Investment fund,Investment / finance,0,https://www.sec.gov/Archives/edgar/data/205080...


### 6.2 Machine Learning on SEC Issuer Names

The live SEC data is used for a lightweight ML filtering process. The model estimates whether a Form D filing is more likely to represent an operating company rather than an investment fund or real-estate/SPV vehicle.

Model setup:

- Input text: issuer name
- Label: weak label from transparent issuer-type rules
- Features: TF-IDF text features
- Model: Logistic Regression

This is weak supervision, not manual ground truth. The goal is to create a practical dashboard filter for recent private-funding filings.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

issuer_ml_df = sec_df[sec_df["issuer_clean"].str.len() > 0].copy()

if issuer_ml_df["likely_operating_company"].nunique() >= 2 and len(issuer_ml_df) >= 12:
    X_issuer_train, X_issuer_test, y_issuer_train, y_issuer_test = train_test_split(
        issuer_ml_df["issuer_clean"],
        issuer_ml_df["likely_operating_company"],
        test_size=0.30,
        random_state=42,
        stratify=issuer_ml_df["likely_operating_company"],
    )
    issuer_model = SkPipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ])
    issuer_model.fit(X_issuer_train, y_issuer_train)
    issuer_pred = issuer_model.predict(X_issuer_test)
    sec_df["operating_company_score"] = issuer_model.predict_proba(sec_df["issuer_clean"])[:, 1]

    print("Weak-label issuer classifier accuracy:", round(accuracy_score(y_issuer_test, issuer_pred), 3))
    print(classification_report(y_issuer_test, issuer_pred, digits=3))

    vectorizer = issuer_model.named_steps["tfidf"]
    clf = issuer_model.named_steps["clf"]
    issuer_terms = pd.DataFrame({
        "term": vectorizer.get_feature_names_out(),
        "coefficient": clf.coef_[0],
    }).sort_values("coefficient", ascending=False)
    display(issuer_terms.head(15))
else:
    sec_df["operating_company_score"] = sec_df["likely_operating_company"]
    print("Not enough class variation for a supervised classifier; using weak label as score.")

sec_df.sort_values("operating_company_score", ascending=False)[
    ["issuer_clean", "issuer_type", "sector_hint", "form_type", "filed_date", "operating_company_score", "url"]
].head(15)

Weak-label issuer classifier accuracy: 0.909
              precision    recall  f1-score   support

           0      0.905     1.000     0.950        19
           1      1.000     0.333     0.500         3

    accuracy                          0.909        22
   macro avg      0.952     0.667     0.725        22
weighted avg      0.918     0.909     0.889        22



,term,coefficient
107,infin8,1.256761
199,spine biopharma,0.725592
217,vascular,0.725592
198,spine,0.725592
27,biopharma,0.725592
32,cagent,0.725592
14,aqua membranes,0.725592
145,membranes,0.725592
13,aqua,0.725592
33,cagent vascular,0.725592


,issuer_clean,issuer_type,sector_hint,form_type,filed_date,operating_company_score,url
86,INFIN8 Inc.,Likely operating company,Other / unclear,D,2026-06-05,0.698118,https://www.sec.gov/Archives/edgar/data/213523...
64,"Water Es Vida Technologies, Inc.",Likely operating company,Technology / analytics,D,2026-06-05,0.698118,https://www.sec.gov/Archives/edgar/data/213876...
21,"Spine BioPharma, Inc.",Likely operating company,Health / industrial science,D,2026-06-05,0.698118,https://www.sec.gov/Archives/edgar/data/185264...
92,"Cagent Vascular, Inc.",Likely operating company,Other / unclear,D,2026-06-05,0.698118,https://www.sec.gov/Archives/edgar/data/161201...
6,"Aqua Membranes, Inc.",Likely operating company,Health / industrial science,D,2026-06-05,0.698118,https://www.sec.gov/Archives/edgar/data/176670...
10,"Aqua Membranes, Inc.",Likely operating company,Health / industrial science,D,2026-06-05,0.698118,https://www.sec.gov/Archives/edgar/data/176670...
14,SignalCraft Analytics Group Inc.,Likely operating company,Technology / analytics,D,2026-06-05,0.694511,https://www.sec.gov/Archives/edgar/data/210993...
17,Partners Group European Direct Lending Strateg...,Investment fund,Investment / finance,D,2026-06-05,0.426248,https://www.sec.gov/Archives/edgar/data/212949...
87,"PG San Antonio OZ, L.P.",Investment fund,Other / unclear,D/A,2026-06-05,0.396898,https://www.sec.gov/Archives/edgar/data/200927...
60,German-Immo Inc,Likely operating company,Other / unclear,D,2026-06-05,0.396898,https://www.sec.gov/Archives/edgar/data/213930...


### 6.3 SEC Filing Theme Discovery

The second ML step clusters issuer names using TF-IDF and K-Means. This is not a deep semantic model, but it gives the dashboard a simple way to summarize current private-filing themes.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD

cluster_df = sec_df[sec_df["issuer_clean"].str.len() > 0].copy()
cluster_count = min(4, max(2, len(cluster_df) // 8))

issuer_vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
X_issuer_topic = issuer_vectorizer.fit_transform(cluster_df["issuer_clean"])

if len(cluster_df) >= 4:
    kmeans = KMeans(n_clusters=cluster_count, random_state=42, n_init=10)
    cluster_df["issuer_cluster"] = kmeans.fit_predict(X_issuer_topic)

    terms = np.array(issuer_vectorizer.get_feature_names_out())
    cluster_terms = []
    for cluster_id in range(cluster_count):
        center = kmeans.cluster_centers_[cluster_id]
        top_terms = terms[np.argsort(center)[-6:]][::-1]
        cluster_terms.append({
            "issuer_cluster": cluster_id,
            "top_terms": ", ".join(top_terms),
            "filing_count": int((cluster_df["issuer_cluster"] == cluster_id).sum()),
        })
    issuer_cluster_terms = pd.DataFrame(cluster_terms)

    svd = TruncatedSVD(n_components=2, random_state=42)
    coords = svd.fit_transform(X_issuer_topic)
    cluster_df["x_topic"] = coords[:, 0]
    cluster_df["y_topic"] = coords[:, 1]
else:
    cluster_df["issuer_cluster"] = 0
    cluster_df["x_topic"] = np.arange(len(cluster_df))
    cluster_df["y_topic"] = 0
    issuer_cluster_terms = pd.DataFrame({"issuer_cluster": [0], "top_terms": ["limited live sample"], "filing_count": [len(cluster_df)]})

display(issuer_cluster_terms)
cluster_df[["issuer_clean", "issuer_type", "sector_hint", "issuer_cluster", "operating_company_score"]].head(20)

,issuer_cluster,top_terms,filing_count
0,0,"llc, series, llc series, heat, solutions, safe...",19
1,1,"harbourvest, vii, investment, harbourvest inve...",9
2,2,"fund lp, lp, fund, pareon, pareon fund, strategy",6
3,3,"lp, fund, fund ii, ii, ii lp, capital",38


,issuer_clean,issuer_type,sector_hint,issuer_cluster,operating_company_score
0,"VIDOR-TX PROPERTY, LLC",Real estate / SPV,Real estate / storage,0,0.330143
1,MT BASE INC.,Likely operating company,Other / unclear,3,0.396898
2,"4PIPO, LLC",Other private issuer,Other / unclear,0,0.270711
3,"TTC MULTI-STRATEGY FUND QP, LP",Investment fund,Investment / finance,2,0.255715
6,"Aqua Membranes, Inc.",Likely operating company,Health / industrial science,3,0.698118
7,Atlas Capital Ex INC.,Investment fund,Investment / finance,3,0.332114
8,Warwick UK Real Estate Fund I,Investment fund,Real estate / storage,3,0.322440
9,"Journey Storage Ventures 01, LLC",Real estate / SPV,Real estate / storage,0,0.324948
10,"Aqua Membranes, Inc.",Likely operating company,Health / industrial science,3,0.698118
11,Macquarie Infrastructure Income Opportunities ...,Investment fund,Investment / finance,3,0.327961


### 6.4 SEC Dashboard Graphs

The following charts are designed for direct dashboard translation. They focus on structured private-filing activity.

In [ ]:
# Interactive Chart 7: Current Form D issuer-type distribution.
issuer_type_summary = sec_df.groupby("issuer_type").agg(
    filings=("filing_id", "nunique"),
    avg_operating_score=("operating_company_score", "mean"),
).reset_index().sort_values("filings", ascending=False)

fig = px.bar(
    issuer_type_summary.sort_values("filings"),
    x="filings",
    y="issuer_type",
    color="avg_operating_score",
    orientation="h",
    text="filings",
    color_continuous_scale=["#60A5FA", "#2DD4BF", "#FF8A3D"],
    hover_data={
        "issuer_type": False,
        "filings": ":,",
        "avg_operating_score": ":.3f",
    },
    labels={
        "filings": "Current Form D filings",
        "issuer_type": "",
        "avg_operating_score": "Avg. operating-company score",
    },
)
fig.update_traces(textposition="outside", cliponaxis=False)
fig = polish_fig(
    fig,
    "SEC Form D Live Filings by Issuer Type",
    "Separates likely operating companies from funds, real-estate/SPVs, and other private issuers.",
    height=500,
)
fig.show()

issuer_type_summary

,issuer_type,filings,avg_operating_score
0,Investment fund,44,0.305450
2,Other private issuer,11,0.312567
1,Likely operating company,9,0.630779
3,Real estate / SPV,8,0.285673


In [ ]:
# Interactive Chart 8: Sector hints among live Form D filings.
sector_sec = sec_df.groupby(["sector_hint", "issuer_type"]).agg(
    filings=("filing_id", "nunique"),
    avg_operating_score=("operating_company_score", "mean"),
).reset_index().sort_values("filings", ascending=False)

fig = px.bar(
    sector_sec,
    x="sector_hint",
    y="filings",
    color="issuer_type",
    hover_data={
        "filings": ":,",
        "avg_operating_score": ":.3f",
        "issuer_type": True,
        "sector_hint": False,
    },
    labels={
        "sector_hint": "Sector hint",
        "filings": "Current Form D filings",
        "issuer_type": "Issuer type",
    },
)
fig = polish_fig(
    fig,
    "SEC Form D Sector Hints",
    "Simple name-based sector hints make live private-filing activity easier to scan in the dashboard.",
    height=580,
)
fig.update_xaxes(tickangle=-25)
fig.show()

sector_sec.head(20)

,sector_hint,issuer_type,filings,avg_operating_score
2,Investment / finance,Investment fund,30,0.292421
3,Other / unclear,Investment fund,11,0.343667
5,Other / unclear,Other private issuer,9,0.320058
0,Energy / climate,Real estate / SPV,5,0.263685
4,Other / unclear,Likely operating company,4,0.547508
1,Health / industrial science,Likely operating company,3,0.698118
7,Real estate / storage,Real estate / SPV,3,0.322318
8,Technology / analytics,Investment fund,2,0.282186
9,Technology / analytics,Likely operating company,2,0.696314
10,Technology / analytics,Other private issuer,2,0.278857


In [ ]:
# Interactive Chart 9: Recent Form D filing activity.
# The SEC current feed often covers a narrow filing window, so this uses an interactive stacked bar.
timeline_df = sec_df.copy()
timeline_df["filed_day"] = pd.to_datetime(timeline_df["filed_date"], errors="coerce").dt.strftime("%Y-%m-%d")
timeline = timeline_df.groupby(["filed_day", "issuer_type"]).agg(
    filings=("filing_id", "nunique"),
    avg_operating_score=("operating_company_score", "mean"),
).reset_index()

fig = px.bar(
    timeline,
    x="filed_day",
    y="filings",
    color="issuer_type",
    hover_data={
        "filings": ":,",
        "avg_operating_score": ":.3f",
        "issuer_type": True,
    },
    labels={
        "filed_day": "Filed date",
        "filings": "Form D filings",
        "issuer_type": "Issuer type",
    },
)
fig = polish_fig(
    fig,
    "Recent SEC Form D Filing Activity",
    "Shows recent private-offering filing volume by issuer type.",
    height=520,
)
fig.update_xaxes(tickangle=-20)
fig.show()

timeline.sort_values(["filed_day", "filings"], ascending=[False, False]).head(20)

,filed_day,issuer_type,filings,avg_operating_score
0,2026-06-05,Investment fund,44,0.305450
2,2026-06-05,Other private issuer,11,0.312567
1,2026-06-05,Likely operating company,9,0.630779
3,2026-06-05,Real estate / SPV,8,0.285673


In [ ]:
# Interactive Chart 10: Operating-company score leaderboard.
leaderboard = sec_df.sort_values("operating_company_score", ascending=False).head(20).copy()
leaderboard["short_issuer"] = leaderboard["issuer_clean"].str.slice(0, 70)

fig = px.scatter(
    leaderboard.sort_values("operating_company_score"),
    x="operating_company_score",
    y="short_issuer",
    color="sector_hint",
    size=np.maximum(leaderboard["size_kb"].fillna(5), 5),
    hover_data={
        "issuer_clean": True,
        "form_type": True,
        "filed_date": True,
        "issuer_type": True,
        "sector_hint": True,
        "url": True,
        "operating_company_score": ":.3f",
        "short_issuer": False,
    },
    labels={
        "operating_company_score": "Operating-company score",
        "short_issuer": "",
        "sector_hint": "Sector hint",
    },
)
fig.update_traces(marker=dict(line=dict(width=1, color="#E5E7EB")))
fig = polish_fig(
    fig,
    "Potential Operating-Company Form D Filings",
    "A review queue for recent private-funding filings that look more like operating companies than funds or SPVs.",
    height=720,
)
fig.update_xaxes(range=[0, 1])
fig.show()

leaderboard[["issuer_clean", "form_type", "filed_date", "issuer_type", "sector_hint", "operating_company_score", "url"]]

,issuer_clean,form_type,filed_date,issuer_type,sector_hint,operating_company_score,url
86,INFIN8 Inc.,D,2026-06-05,Likely operating company,Other / unclear,0.698118,https://www.sec.gov/Archives/edgar/data/213523...
64,"Water Es Vida Technologies, Inc.",D,2026-06-05,Likely operating company,Technology / analytics,0.698118,https://www.sec.gov/Archives/edgar/data/213876...
21,"Spine BioPharma, Inc.",D,2026-06-05,Likely operating company,Health / industrial science,0.698118,https://www.sec.gov/Archives/edgar/data/185264...
92,"Cagent Vascular, Inc.",D,2026-06-05,Likely operating company,Other / unclear,0.698118,https://www.sec.gov/Archives/edgar/data/161201...
6,"Aqua Membranes, Inc.",D,2026-06-05,Likely operating company,Health / industrial science,0.698118,https://www.sec.gov/Archives/edgar/data/176670...
10,"Aqua Membranes, Inc.",D,2026-06-05,Likely operating company,Health / industrial science,0.698118,https://www.sec.gov/Archives/edgar/data/176670...
14,SignalCraft Analytics Group Inc.,D,2026-06-05,Likely operating company,Technology / analytics,0.694511,https://www.sec.gov/Archives/edgar/data/210993...
17,Partners Group European Direct Lending Strateg...,D,2026-06-05,Investment fund,Investment / finance,0.426248,https://www.sec.gov/Archives/edgar/data/212949...
87,"PG San Antonio OZ, L.P.",D/A,2026-06-05,Investment fund,Other / unclear,0.396898,https://www.sec.gov/Archives/edgar/data/200927...
60,German-Immo Inc,D,2026-06-05,Likely operating company,Other / unclear,0.396898,https://www.sec.gov/Archives/edgar/data/213930...


In [ ]:
# Interactive Chart 11: SEC issuer-name theme map.
topic_plot = cluster_df.copy()
topic_plot["issuer_cluster"] = topic_plot["issuer_cluster"].astype(str)
topic_plot["display_size"] = np.maximum(topic_plot["operating_company_score"].fillna(0), 0.1)

fig = px.scatter(
    topic_plot,
    x="x_topic",
    y="y_topic",
    color="issuer_cluster",
    size="display_size",
    hover_data={
        "issuer_clean": True,
        "issuer_type": True,
        "sector_hint": True,
        "operating_company_score": ":.3f",
        "x_topic": False,
        "y_topic": False,
        "display_size": False,
        "issuer_cluster": True,
    },
    labels={"issuer_cluster": "Issuer-name cluster"},
)
fig = polish_fig(
    fig,
    "SEC Form D Issuer Theme Map",
    "TF-IDF + K-Means clustering summarizes current private-filing issuer-name themes.",
    height=620,
)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
fig.show()

issuer_cluster_terms

,issuer_cluster,top_terms,filing_count
0,0,"llc, series, llc series, heat, solutions, safe...",19
1,1,"harbourvest, vii, investment, harbourvest inve...",9
2,2,"fund lp, lp, fund, pareon, pareon fund, strategy",6
3,3,"lp, fund, fund ii, ii, ii lp, capital",38


In [ ]:
# Dashboard-ready summary tables for the SEC live enrichment section.
sec_dashboard_summary = sec_df.groupby(["issuer_type", "sector_hint"]).agg(
    filings=("filing_id", "nunique"),
    avg_operating_score=("operating_company_score", "mean"),
    amendments=("is_amendment", "sum"),
).reset_index().sort_values(["filings", "avg_operating_score"], ascending=False)

sec_review_queue = sec_df.sort_values("operating_company_score", ascending=False)[
    ["issuer_clean", "form_type", "filed_date", "issuer_type", "sector_hint", "operating_company_score", "url"]
].head(25)

display(Markdown("### SEC live filing summary"))
display(sec_dashboard_summary)
display(Markdown("### Potential operating-company review queue"))
display(sec_review_queue)

### SEC live filing summary

,issuer_type,sector_hint,filings,avg_operating_score,amendments
0,Investment fund,Investment / finance,30,0.292421,21
1,Investment fund,Other / unclear,11,0.343667,4
7,Other private issuer,Other / unclear,9,0.320058,2
9,Real estate / SPV,Energy / climate,5,0.263685,0
5,Likely operating company,Other / unclear,4,0.547508,0
4,Likely operating company,Health / industrial science,3,0.698118,0
10,Real estate / SPV,Real estate / storage,3,0.322318,1
6,Likely operating company,Technology / analytics,2,0.696314,0
3,Investment fund,Technology / analytics,2,0.282186,0
8,Other private issuer,Technology / analytics,2,0.278857,2


### Potential operating-company review queue

,issuer_clean,form_type,filed_date,issuer_type,sector_hint,operating_company_score,url
86,INFIN8 Inc.,D,2026-06-05,Likely operating company,Other / unclear,0.698118,https://www.sec.gov/Archives/edgar/data/213523...
64,"Water Es Vida Technologies, Inc.",D,2026-06-05,Likely operating company,Technology / analytics,0.698118,https://www.sec.gov/Archives/edgar/data/213876...
21,"Spine BioPharma, Inc.",D,2026-06-05,Likely operating company,Health / industrial science,0.698118,https://www.sec.gov/Archives/edgar/data/185264...
92,"Cagent Vascular, Inc.",D,2026-06-05,Likely operating company,Other / unclear,0.698118,https://www.sec.gov/Archives/edgar/data/161201...
6,"Aqua Membranes, Inc.",D,2026-06-05,Likely operating company,Health / industrial science,0.698118,https://www.sec.gov/Archives/edgar/data/176670...
10,"Aqua Membranes, Inc.",D,2026-06-05,Likely operating company,Health / industrial science,0.698118,https://www.sec.gov/Archives/edgar/data/176670...
14,SignalCraft Analytics Group Inc.,D,2026-06-05,Likely operating company,Technology / analytics,0.694511,https://www.sec.gov/Archives/edgar/data/210993...
17,Partners Group European Direct Lending Strateg...,D,2026-06-05,Investment fund,Investment / finance,0.426248,https://www.sec.gov/Archives/edgar/data/212949...
87,"PG San Antonio OZ, L.P.",D/A,2026-06-05,Investment fund,Other / unclear,0.396898,https://www.sec.gov/Archives/edgar/data/200927...
60,German-Immo Inc,D,2026-06-05,Likely operating company,Other / unclear,0.396898,https://www.sec.gov/Archives/edgar/data/213930...


## 7. Integrated Findings for FounderRadar

The project now contains two complementary analytical layers:

1. **Historical layer:** Kaggle Startup Investments is used to model recorded exits using funding, market, geography, and founder/team signals.
2. **Live private-funding layer:** SEC Form D current filings provide recent structured signals of private capital activity.

### Main findings

- The historical model supports the core FounderRadar idea: funding maturity is important, but sector, geography, founder count, and education coverage add useful context.
- SEC Form D is a more defensible live-data source than general news because it is structured and directly connected to private offerings.
- The SEC feed contains many funds, SPVs, and real-estate vehicles; therefore filtering is required before it is useful for startup discovery.
- A lightweight issuer-name classifier can create a review queue of filings that appear more likely to represent operating companies.
- Issuer-name clustering provides a simple way to summarize current private-filing themes.

### Dashboard implementation recommendation

Use the Kaggle model for the main historical FounderRadar analysis, then add SEC Form D as a secondary live panel:

- Recent Form D filing feed
- Issuer type
- Sector hint
- Operating-company score
- Filing URL
- Filing timeline

This is feasible for a student-led Python Shiny project because the SEC feed can be cached and refreshed periodically.

## 8. Narrative for Slides and Progress Report

**Main narrative:**

> FounderRadar combines historical startup investment outcomes with founder/team signals and live SEC private-offering filings to support investor-oriented startup discovery.

**Suggested storyline:**

1. **Historical investment context:** Funding amount and funding maturity are important predictors of recorded exit outcomes.
2. **FounderRadar contribution:** The dashboard adds founder/team context, including known founder count, education coverage, prior company exposure, sector, and geography.
3. **Live data extension:** SEC Form D filings add a near-real-time view of recent private capital activity.
4. **Machine learning contribution:** The notebook applies machine learning in two places: historical exit prediction and live issuer filtering/theme discovery.
5. **Investor value:** Users can compare historical success patterns while monitoring recent private-funding signals.

**Limitations:**

- The Kaggle Startup Investments dataset is historical and may be incomplete.
- `success_exit` captures acquisition and IPO-style outcomes, but does not capture all forms of startup success.
- Founder education and prior-role records are sparse.
- SEC Form D is U.S.-centric and includes many funds, SPVs, and real-estate vehicles.
- The SEC issuer classifier uses weak supervision and should support filtering rather than final investment decisions.

**Recommended dashboard components:**

- KPI cards: number of companies, known founder records, education records, exits, and median funding
- Founder signal summary: exit rate by team, education, and funding maturity segments
- Sector landscape: market category, funding intensity, and exit rate
- Geographic ecosystem view: country-level company count and exit rate
- Machine learning insight panel: predicted exit probability and top contributing signal groups
- Live SEC Form D panel: recent private-offering filings ranked by operating-company score